# Notebook 2 — traditional scheduler comparison

Each panel shows **service order**: photo 1 is served first. The numbers are ranks, not image priority classes. FIFO, round robin, EDF, PF and EXP/PF use queue/timing/channel metadata only. Fixed priority is shown twice: a synthetic network QoS priority baseline and, optionally, the selected image-model priority. No scheduler infers image meaning.

This is a reproducible toy queue comparison. Synthetic radio rates do not make it a validated 5G network simulator.

In [ ]:
# Set REPO_URL only in a temporary Colab/Kaggle runtime. Local Jupyter uses the existing checkout.
REPO_URL = None  # e.g. "https://github.com/your-name/SemanticSchedulerEnd2End.git"
REPO_COMMIT = None  # Pin a commit for repeatable cloud runs; None checks out remote HEAD.
PERSISTENT_RESULTS = None  # REQUIRED in cloud: mounted Drive/Kaggle persistent output path.

import os, sys, subprocess, shutil
from pathlib import Path
def cloud_runtime():
    return 'COLAB_GPU' in os.environ or 'KAGGLE_KERNEL_RUN_TYPE' in os.environ
if cloud_runtime():
    if not REPO_URL or not PERSISTENT_RESULTS:
        raise ValueError('Set REPO_URL and PERSISTENT_RESULTS before running in Colab/Kaggle')
    checkout = Path('/tmp/semantic_scheduler_pilot_checkout').resolve()
    assert checkout == Path('/tmp/semantic_scheduler_pilot_checkout').resolve()
    if checkout.exists():
        shutil.rmtree(checkout)  # Only this fixed disposable runtime checkout is reset.
    subprocess.run(['git', 'clone', REPO_URL, str(checkout)], check=True)
    if REPO_COMMIT:
        subprocess.run(['git', '-C', str(checkout), 'checkout', REPO_COMMIT], check=True)
    PROJECT_ROOT = checkout
else:
    here = Path.cwd().resolve()
    PROJECT_ROOT = next((p for p in [here, *here.parents] if (p/'PROJECT_MAP.md').is_file()), None)
    if PROJECT_ROOT is None:
        raise RuntimeError('Start Jupyter from this repository or a subdirectory')
sys.path.insert(0, str(PROJECT_ROOT))
RESULTS = Path(PERSISTENT_RESULTS).expanduser().resolve() if PERSISTENT_RESULTS else PROJECT_ROOT/'results'/'scheduler_pilot'
RESULTS.mkdir(parents=True, exist_ok=True)
MODEL_ROOT = PROJECT_ROOT/'models'/'priority_classifiers'
try:
    COMMIT = subprocess.check_output(['git','-C',str(PROJECT_ROOT),'rev-parse','HEAD'], text=True).strip()
except Exception:
    COMMIT = 'unavailable'
print('Repository:', PROJECT_ROOT, 'Commit:', COMMIT, 'Results:', RESULTS)


In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from PIL import Image
manifest_path = RESULTS/'pilot_manifest.csv'
if not manifest_path.exists():
    raise FileNotFoundError(f'Run Notebook 1 sample cell first: {manifest_path}')
photos = pd.read_csv(manifest_path).head(15).copy()
if photos.empty: raise ValueError('Pilot manifest is empty')
for path in photos.image_path:
    if not Path(path).is_file(): raise FileNotFoundError(path)
seed = 42
rng = np.random.default_rng(seed)
photos['arrival_ms'] = np.arange(len(photos))*2  # Deliberate queue contention.
photos['deadline_ms'] = photos['arrival_ms'] + rng.integers(30, 130, len(photos))
photos['ue_id'] = np.arange(len(photos)) % 4
photos['qos_priority'] = rng.integers(0, 4, len(photos))  # Independent of image content.
photos['channel_rate'] = rng.uniform(0.5, 2.0, len(photos))  # Synthetic radio rate.
photos['avg_throughput'] = rng.uniform(0.5, 1.5, len(photos))
photos['resource_units'] = rng.integers(1, 4, len(photos))
display(photos[['image_id','arrival_ms','deadline_ms','ue_id','qos_priority','channel_rate','resource_units']])


In [ ]:
# Rank one pending image at a time. All algorithms receive identical jobs.
# Round robin cycles users; PF uses instantaneous rate / average throughput;
# EXP/PF combines head-of-line waiting time with the PF rate ratio.
def order_jobs(frame, method, priority_column='qos_priority'):
    waiting = frame.to_dict('records')
    completed, last_ue, now = [], -1, 0.0
    avg = {int(ue): 1.0 for ue in frame.ue_id.unique()}
    while waiting:
        eligible = [r for r in waiting if r['arrival_ms'] <= now]
        if not eligible:
            now = min(r['arrival_ms'] for r in waiting)
            eligible = [r for r in waiting if r['arrival_ms'] <= now]
        if method == 'FIFO':
            pick = min(eligible, key=lambda r:(r['arrival_ms'], r['image_id']))
        elif method == 'Round Robin':
            ue_list = sorted({int(r['ue_id']) for r in eligible})
            next_ue = next((ue for ue in ue_list if ue > last_ue), ue_list[0])
            pick = min((r for r in eligible if int(r['ue_id']) == next_ue), key=lambda r:r['arrival_ms'])
        elif method == 'Fixed Priority':
            pick = min(eligible, key=lambda r:(r[priority_column],r['arrival_ms']))
        elif method == 'EDF':
            pick = min(eligible, key=lambda r:(r['deadline_ms'],r['arrival_ms']))
        elif method in ('PF','EXP/PF'):
            def score(r):
                fairness = r['channel_rate']/avg[int(r['ue_id'])]
                if method == 'PF': return fairness
                wait = max(0., now-r['arrival_ms'])
                mean_wait = np.mean([max(0., now-x['arrival_ms']) for x in eligible])
                exponent = (wait-mean_wait)/(1.+np.sqrt(mean_wait))
                return np.exp(np.clip(exponent, -10., 10.))*fairness
            pick = max(eligible, key=lambda r:(score(r),-r['arrival_ms']))
        else:
            raise ValueError(method)
        waiting.remove(pick); completed.append(pick)
        now += 10*pick['resource_units']/pick['channel_rate']
        ue = int(pick['ue_id'])
        avg[ue] = 0.9*avg[ue] + 0.1*pick['channel_rate']
        last_ue = ue
    return completed

orders = {name:order_jobs(photos,name) for name in ['FIFO','Round Robin','Fixed Priority','EDF','PF','EXP/PF']}


In [ ]:
# Optional semantic-priority branch: reads only model CSV, never the human ground truth.
SELECTED_MODEL = None  # Same model chosen in Notebook 1, if you want a semantic-priority panel.
if SELECTED_MODEL:
    path = RESULTS/f'{SELECTED_MODEL}_priority_labels.csv'
    labels = pd.read_csv(path)[['image_id','priority_class','status']]
    semantic = photos.merge(labels, on='image_id', validate='one_to_one')
    if len(semantic) != len(photos) or semantic.priority_class.isna().any() or not semantic.status.eq('ok').all():
        raise ValueError('Selected model has missing/unmapped/failed priorities; inspect CSV before scheduling')
    orders['Image Priority'] = order_jobs(semantic,'Fixed Priority','priority_class')

def show_order(name, arranged):
    fig, axes = plt.subplots(3, 5, figsize=(15, 9))
    for rank, (ax, row) in enumerate(zip(axes.flat, arranged), start=1):
        ax.imshow(Image.open(row['image_path']).resize((320, 220)))
        ax.set_title(f'{rank}. {row["image_id"]}', fontsize=11)
        ax.axis('off')
    for ax in list(axes.flat)[len(arranged):]: ax.axis('off')
    fig.suptitle(f'{name}: service order, 1 = served first')
    fig.tight_layout(); fig.savefig(RESULTS/f'order_{name.lower().replace("/","_").replace(" ","_")}.png',dpi=150)
    plt.show()
for name, arranged in orders.items():
    print(name, 'order:', [r['image_id'] for r in arranged])
    show_order(name, arranged)


**Interpretation:** Traditional schedulers need externally supplied metadata. These panels compare ordering, not image classification accuracy. Add manually reviewed priorities before calculating urgency delay or missed critical deadlines as research metrics.